# 1. SupportSense — Exploratory Data Analysis
### Getting started; opening and exploring the data

**Business case:** Northstar Cloud is a fictional B2B SaaS company. SupportSense explores whether a document-grounded assistant can help support agents answer common product, billing, security, and troubleshooting questions.

This first notebook follows a practical EDA workflow: load the source files, inspect structure and data quality, examine the knowledge base, explore the evaluation questions, and translate findings into business questions for the modeling notebooks.

**Important data boundary:** this repository contains a small synthetic knowledge base and a synthetic evaluation set. It does **not** contain real customer tickets, customer identifiers, satisfaction scores, or historical handling times. Therefore, this notebook analyzes the available documents and benchmark questions rather than inventing customer or ticket trends.

## 1. Load the data

The project stores approved knowledge-base documents and benchmark questions as JSON files under `data/`. The path helper below works when the notebook is launched either from the repository root or from the `notebooks/` folder.

In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve project root whether Jupyter starts in the repository root or notebooks/
CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in CANDIDATE_ROOTS if (p / "data" / "company_docs.json").exists()), None)
if ROOT is None:
    raise FileNotFoundError(
        "Could not find data/company_docs.json. Start Jupyter from the repository root "
        "or from its notebooks/ folder."
    )

docs_path = ROOT / "data" / "company_docs.json"
questions_path = ROOT / "data" / "evaluation_questions.json"

with docs_path.open(encoding="utf-8") as f:
    documents_raw = json.load(f)
with questions_path.open(encoding="utf-8") as f:
    questions_raw = json.load(f)

docs = pd.DataFrame(documents_raw)
questions = pd.DataFrame(questions_raw)

print(f"Project root: {ROOT}")
print(f"Knowledge-base documents: {len(docs):,}")
print(f"Evaluation questions: {len(questions):,}")

display(docs.head())
display(questions.head())

## 2. Get to know the data

Review column types and missing values before analyzing the records. The knowledge-base table should contain document identifiers, titles, and text. The question table includes expected source labels and answerability labels for offline evaluation.

In [ ]:
print("KNOWLEDGE-BASE SCHEMA")
docs.info()

print("\\nEVALUATION-QUESTION SCHEMA")
questions.info()

In [ ]:
print("DOCUMENT DATA — DESCRIPTIVE SUMMARY")
display(docs.describe(include="all").T)

print("QUESTION DATA — DESCRIPTIVE SUMMARY")
display(questions.describe(include="all").T)

## 3. Handle missing values, duplicates, and anomalies

For this small project, quality checks are more useful than blindly dropping records. Missing expected documents are valid for unanswerable questions, so they must not be treated as data errors. Document IDs and question IDs, however, should be present and unique.

In [ ]:
def quality_report(frame, name):
    report = pd.DataFrame({
        "missing_count": frame.isna().sum(),
        "missing_pct": (frame.isna().mean() * 100).round(2),
        "dtype": frame.dtypes.astype(str)
    })
    print(f"{name}: {len(frame):,} rows, {frame.shape[1]} columns")
    print(f"Exact duplicate rows: {frame.astype(str).duplicated().sum()}")
    display(report)

quality_report(docs, "Knowledge base")
quality_report(questions, "Evaluation questions")

In [ ]:
# Validate required fields and identifier uniqueness without silently deleting data.
doc_required = ["doc_id", "title", "text"]
question_required = ["question_id", "question", "answerable", "expected_doc", "expected_terms"]

print("Missing required document fields:")
display(docs[docs[doc_required].isna().any(axis=1)])

print("Missing required question fields (expected_doc may be null for unanswerable questions):")
display(questions[questions[[c for c in question_required if c != "expected_doc"]].isna().any(axis=1)])

print("Duplicate document IDs:", docs["doc_id"].duplicated().sum())
print("Duplicate question IDs:", questions["question_id"].duplicated().sum())
print("Duplicate document texts:", docs["text"].duplicated().sum())
print("Duplicate question texts:", questions["question"].duplicated().sum())

In [ ]:
# Work on analysis copies and normalize text fields conservatively.
docs_eda = docs.copy()
questions_eda = questions.copy()

for col in ["doc_id", "title", "text"]:
    docs_eda[col] = docs_eda[col].astype("string").str.strip()

for col in ["question_id", "question"]:
    questions_eda[col] = questions_eda[col].astype("string").str.strip()

questions_eda["answerable"] = questions_eda["answerable"].astype("boolean")

# Report rather than remove potential duplicate records.
duplicate_doc_rows = docs_eda[docs_eda.duplicated(subset=["doc_id"], keep=False)].sort_values("doc_id")
duplicate_question_rows = questions_eda[
    questions_eda.duplicated(subset=["question_id"], keep=False)
].sort_values("question_id")

print(f"Rows with repeated doc_id: {len(duplicate_doc_rows)}")
print(f"Rows with repeated question_id: {len(duplicate_question_rows)}")

## 4. Create analysis features

Document and question length, word counts, and simple topic labels help describe the coverage of this small corpus. Topic labels are assigned from document titles using an explicit mapping for exploratory reporting; they are not learned categories.

In [ ]:
docs_eda["title_word_count"] = docs_eda["title"].str.split().str.len()
docs_eda["text_char_count"] = docs_eda["text"].str.len()
docs_eda["text_word_count"] = docs_eda["text"].str.split().str.len()

topic_map = {
    "Password reset": "Account access & security",
    "Billing invoices": "Billing & subscriptions",
    "Refund policy": "Billing & subscriptions",
    "Two-factor authentication": "Account access & security",
    "API rate limits": "API & technical support",
    "Data export": "Data management",
    "Service status": "Service reliability",
    "Account permissions": "Account access & security",
    "Cancel subscription": "Billing & subscriptions",
    "Security incident reporting": "Account access & security",
}
docs_eda["topic"] = docs_eda["title"].map(topic_map).fillna("Other / review")

questions_eda["question_char_count"] = questions_eda["question"].str.len()
questions_eda["question_word_count"] = questions_eda["question"].str.split().str.len()
questions_eda["expected_terms_count"] = questions_eda["expected_terms"].apply(
    lambda value: len(value) if isinstance(value, list) else 0
)

display(docs_eda[["doc_id", "title", "topic", "text_word_count", "text_char_count"]])
display(questions_eda[["question_id", "question", "answerable", "expected_doc", "question_word_count"]])

## 5. General analysis of the knowledge base

### Analysis question
**Which support topics are covered by the current knowledge base, and where is the documentation concentrated?**

This is an inventory of the content available to the assistant. It should not be interpreted as the frequency of customer issues, because the dataset contains one curated document per topic rather than real ticket volumes.

In [ ]:
topic_summary = (
    docs_eda.groupby("topic", dropna=False)
    .agg(
        document_count=("doc_id", "nunique"),
        avg_words=("text_word_count", "mean"),
        total_words=("text_word_count", "sum")
    )
    .sort_values("document_count", ascending=False)
    .reset_index()
)
topic_summary["avg_words"] = topic_summary["avg_words"].round(1)
display(topic_summary)

fig, ax = plt.subplots(figsize=(10, 5))
plot_data = topic_summary.sort_values("document_count")
ax.barh(plot_data["topic"], plot_data["document_count"])
ax.set_title("Knowledge-base coverage by support topic")
ax.set_xlabel("Number of documents")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

**Interpretation:** the chart shows documentation coverage, not demand. A topic with fewer documents may still generate many support contacts. To estimate demand, a future version should add a real ticket dataset with issue category and created timestamp.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_data = docs_eda.sort_values("text_word_count")
ax.barh(plot_data["title"], plot_data["text_word_count"])
ax.set_title("Document length by knowledge-base article")
ax.set_xlabel("Word count")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

print("Document word-count summary:")
display(docs_eda["text_word_count"].describe().to_frame("word_count"))

## 6. Evaluation-question analysis

### Analysis question
**Does the benchmark test both questions answerable from approved documents and questions that should be escalated or answered cautiously?**

The `answerable` label is a designed benchmark label. It represents the expected handling policy for each example; it is not a model prediction.

In [ ]:
answerability_summary = (
    questions_eda["answerable"]
    .map({True: "Answerable from current documents", False: "Not fully answerable"})
    .value_counts(dropna=False)
    .rename_axis("benchmark_group")
    .reset_index(name="question_count")
)
answerability_summary["share_pct"] = (
    answerability_summary["question_count"] / len(questions_eda) * 100
).round(1)
display(answerability_summary)

fig, ax = plt.subplots(figsize=(7, 4))
labels = answerability_summary["benchmark_group"]
values = answerability_summary["question_count"]
ax.bar(labels, values)
ax.set_title("Evaluation questions by expected answerability")
ax.set_ylabel("Number of questions")
ax.tick_params(axis="x", rotation=12)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Count which expected source documents are tested.
expected_doc_counts = (
    questions_eda.assign(expected_doc_label=questions_eda["expected_doc"].fillna("No direct source"))
    ["expected_doc_label"].value_counts()
    .rename_axis("expected_source")
    .reset_index(name="question_count")
)
display(expected_doc_counts)

fig, ax = plt.subplots(figsize=(9, 5))
plot_data = expected_doc_counts.sort_values("question_count")
ax.barh(plot_data["expected_source"], plot_data["question_count"])
ax.set_title("Benchmark coverage by expected source document")
ax.set_xlabel("Number of questions")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(questions_eda["question_word_count"], bins=range(0, int(questions_eda["question_word_count"].max()) + 3, 2), edgecolor="black")
ax.set_title("Distribution of evaluation-question length")
ax.set_xlabel("Words per question")
ax.set_ylabel("Question count")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

display(questions_eda[["question_id", "question_word_count", "answerable", "expected_terms_count"]].sort_values("question_word_count", ascending=False))

## 7. Map benchmark questions to business topics

This joins each question's expected source document to the knowledge-base inventory. Unanswerable questions without an expected source remain labelled as requiring verification or escalation.

In [ ]:
doc_lookup = docs_eda[["doc_id", "title", "topic"]].rename(
    columns={"doc_id": "expected_doc", "title": "expected_title", "topic": "expected_topic"}
)
question_coverage = questions_eda.merge(doc_lookup, on="expected_doc", how="left")
question_coverage["expected_topic"] = question_coverage["expected_topic"].fillna("Needs verification / escalation")

display(question_coverage[
    ["question_id", "question", "answerable", "expected_doc", "expected_title", "expected_topic"]
])

In [ ]:
topic_question_summary = (
    question_coverage.groupby("expected_topic", dropna=False)
    .agg(
        question_count=("question_id", "nunique"),
        answerable_count=("answerable", lambda s: int(s.fillna(False).sum())),
        unanswerable_count=("answerable", lambda s: int((~s.fillna(False)).sum()))
    )
    .sort_values("question_count", ascending=False)
    .reset_index()
)
display(topic_question_summary)

## 8. Data quality checks for the next modeling stage

The next notebook evaluates retrieval. Before that, verify the benchmark's structural assumptions: source IDs should resolve to documents for answerable questions, while unanswerable examples may intentionally have no expected source.

In [ ]:
known_doc_ids = set(docs_eda["doc_id"].dropna().astype(str))
answerable_missing_source = questions_eda[
    questions_eda["answerable"].fillna(False) & ~questions_eda["expected_doc"].isin(known_doc_ids)
]
unanswerable_with_source = questions_eda[
    (~questions_eda["answerable"].fillna(False)) & questions_eda["expected_doc"].notna()
]

print(f"Answerable questions whose expected document is missing: {len(answerable_missing_source)}")
print(f"Unanswerable questions that still have an expected source label: {len(unanswerable_with_source)}")
print(f"Unique document IDs: {docs_eda['doc_id'].nunique()} / {len(docs_eda)}")
print(f"Unique question IDs: {questions_eda['question_id'].nunique()} / {len(questions_eda)}")

if len(answerable_missing_source):
    display(answerable_missing_source)

## 9. Initial business findings and questions

### What this dataset supports
- The knowledge base covers account access/security, billing/subscriptions, API/technical support, data management, and service reliability.
- The benchmark contains both answerable and deliberately unanswerable examples, so later evaluation can consider abstention as well as answer quality.
- Some source documents are tested by multiple questions, while other documents receive less benchmark attention. Retrieval results should therefore be inspected question by question.
- Document length and topic counts describe the knowledge base, not real-world ticket demand, agent performance, or customer satisfaction.

### Business questions to carry forward
1. Can retrieval find the expected approved document for each benchmark question?
2. Does the RAG treatment abstain when evidence is insufficient?
3. Are answers grounded in source text and do citations identify the right documents?
4. Does an AI-assisted workflow improve quality enough to justify added latency or operational cost?
5. What additional real-world data is required before a live A/B test or savings claim?

**Next notebook:** *SupportSense — Document Retrieval*, where we measure top-k source retrieval and inspect failure cases.